In [1]:
import os
import shutil
import pandas as pd # Used for potentially reading and combining later, but for now just for context

def consolidate_summary_csvs(base_experiment_dir, target_summary_dir_name="summary_statistics_consolidated"):
    """
    Copies specific CSV files from encoder-specific subdirectories into a single
    target summary directory.

    Args:
        base_experiment_dir (str): The path to the base directory containing
                                   encoder-specific subfolders.
        target_summary_dir_name (str, optional): The name of the directory
                                                 to create within base_experiment_dir
                                                 for storing consolidated CSVs.
                                                 Defaults to "summary_statistics_consolidated".
    """
    print(f"Starting CSV consolidation process...")
    print(f"Base experiment directory: {base_experiment_dir}")

    # Define the target directory path
    target_dir = os.path.join(base_experiment_dir, target_summary_dir_name)

    # Create the target directory if it doesn't exist
    try:
        os.makedirs(target_dir, exist_ok=True)
        print(f"Ensured target directory exists: {target_dir}")
    except OSError as e:
        print(f"Error creating target directory {target_dir}: {e}")
        return

    # CSV files to look for in each encoder directory
    csv_files_to_copy = [
        "mlp_geoshapley_summary_statistics.csv",
        "xgb_geoshapley_summary_statistics.csv",
        "model_metrics.csv"
    ]

    copied_files_count = 0
    processed_encoder_dirs = 0

    # Iterate through items in the base experiment directory
    if not os.path.isdir(base_experiment_dir):
        print(f"Error: Base experiment directory '{base_experiment_dir}' not found or is not a directory.")
        return

    for item_name in os.listdir(base_experiment_dir):
        encoder_dir_path = os.path.join(base_experiment_dir, item_name)

        # Check if the item is a directory and not the target summary directory itself
        if os.path.isdir(encoder_dir_path) and item_name != target_summary_dir_name:
            encoder_type = item_name # The directory name is the encoder type
            print(f"\nProcessing encoder directory: {encoder_type}")
            processed_encoder_dirs += 1

            for csv_filename in csv_files_to_copy:
                source_file_path = os.path.join(encoder_dir_path, csv_filename)

                if os.path.exists(source_file_path):
                    # Construct a new filename to include the encoder type
                    # e.g., "Space2Vec-theory_mlp_geoshapley_summary_statistics.csv"
                    # For brevity, we can also shorten the original filename if desired
                    if csv_filename == "mlp_geoshapley_summary_statistics.csv":
                        new_filename = f"{encoder_type}_mlp_summary_stats.csv"
                    elif csv_filename == "xgb_geoshapley_summary_statistics.csv":
                        new_filename = f"{encoder_type}_xgb_summary_stats.csv"
                    elif csv_filename == "model_metrics.csv":
                        new_filename = f"{encoder_type}_model_metrics.csv"
                    else:
                        # Fallback for any other CSVs, though we've listed them explicitly
                        base, ext = os.path.splitext(csv_filename)
                        new_filename = f"{encoder_type}_{base}{ext}"

                    destination_file_path = os.path.join(target_dir, new_filename)

                    try:
                        shutil.copy2(source_file_path, destination_file_path)
                        print(f"  Copied: {csv_filename} -> {new_filename}")
                        copied_files_count += 1
                    except Exception as e:
                        print(f"  Error copying {source_file_path} to {destination_file_path}: {e}")
                else:
                    print(f"  Not found: {csv_filename} in {encoder_type}")

    if processed_encoder_dirs == 0:
        print("\nNo encoder subdirectories found to process.")
    else:
        print(f"\nConsolidation complete. Copied {copied_files_count} CSV files into '{target_dir}'.")

if __name__ == "__main__":
    # --- Configuration ---
    # This should match the BASE_EXPERIMENT_DIR from your main script
    BASE_DIR = './results/dumb_multi_embedding_fixed'
    # Name of the directory where all summary CSVs will be copied
    SUMMARY_DIR_NAME = "all_summary_statistics_consolidated"

    # Check if the base directory exists before running
    if not os.path.exists(BASE_DIR):
        print(f"Error: The specified base directory '{BASE_DIR}' does not exist.")
        print("Please ensure this path is correct and the experiments have been run.")
    else:
        consolidate_summary_csvs(BASE_DIR, SUMMARY_DIR_NAME)

Starting CSV consolidation process...
Base experiment directory: ./results/dumb_multi_embedding_fixed
Ensured target directory exists: ./results/dumb_multi_embedding_fixed/all_summary_statistics_consolidated

Processing encoder directory: wrap_ffn
  Copied: mlp_geoshapley_summary_statistics.csv -> wrap_ffn_mlp_summary_stats.csv
  Copied: xgb_geoshapley_summary_statistics.csv -> wrap_ffn_xgb_summary_stats.csv
  Copied: model_metrics.csv -> wrap_ffn_model_metrics.csv

Processing encoder directory: Sphere2Vec-sphereM
  Copied: mlp_geoshapley_summary_statistics.csv -> Sphere2Vec-sphereM_mlp_summary_stats.csv
  Copied: xgb_geoshapley_summary_statistics.csv -> Sphere2Vec-sphereM_xgb_summary_stats.csv
  Copied: model_metrics.csv -> Sphere2Vec-sphereM_model_metrics.csv

Processing encoder directory: rff
  Copied: mlp_geoshapley_summary_statistics.csv -> rff_mlp_summary_stats.csv
  Copied: xgb_geoshapley_summary_statistics.csv -> rff_xgb_summary_stats.csv
  Copied: model_metrics.csv -> rff_mode